# Imports

In [2]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 

from datetime import datetime 
import xgboost as xgb

import os

from sklearn.metrics import f1_score, classification_report
import gc
from sklearn.impute import SimpleImputer

In [30]:
data = ['transaction_id', 'is_fraud', 'created_at', 'is_subscription', 'transaction_type',
        'currency_amount', 'currency_id', 'merchant_customer_id',
        'merchant_customer_email', 'merchant_country', 'merchant_language', 'ip_address',
        'platform', 'merchant_id', 'merchant_shop_id', 'merchant_shop_name', 'is_secured',
        'ip_country', 'payment_type', 'user_agent', 'card_id', 'bank', 'cardbrand',
        'cardcountry', 'cardtype', 'bin', 'card_exp_relative', 'card_holder_first_name',
        'card_holder_last_name']

In [31]:
train_path = '/kaggle/input/int20h_test_2025/train.csv' 
test_path = '/kaggle/input/int20h_test_2025/test.csv'

# EDA

In [ ]:
total_rows = sum(1 for _ in open(train_path)) - 1  
indices_to_skip = set(np.random.choice(
    range(1, total_rows + 1),  
    size=int(total_rows * 0.58),  
    replace=False
))

train = pd.read_csv(train_path, 
                 skiprows=lambda x: x in indices_to_skip, usecols=data)

In [6]:
train.head()

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,transaction_id,is_fraud,currency_amount,currency_id,amount_scaled,merchant_id,merchant_shop_id,order_number,ip_country,payment_type,traffic_source,transaction_source,user_agent,device,bin,card_exp_relative
0,2023059597300930765,0,1348.65,2,8,16325335396072746098,15442,NaN,USA,NaN,d4ffd8339ee260e59f4f461700e9b1d885de372a6fdb71...,NaN,NaN,NaN,82178912e9cb7cdfe4397e288172d2ddb5b4ca24de4ad7...,59.0
1,1582863622564573009,0,201.15,4,98,14575487352341610617,14942,NaN,AUS,NaN,NaN,NaN,Mozilla/5.0 (iPhone; CPU iPhone OS 17_3 like M...,NaN,c84a4f90930cf35cb2718e49919df4f2400347c3e28e5c...,46.0
2,15650942020679503118,0,201.15,4,99,14575487352341610617,14942,NaN,AUS,NaN,NaN,NaN,Mozilla/5.0 (iPhone; CPU iPhone OS 17_2_1 like...,NaN,a321caacb48f0dd4ac473a76fb7a8eaa54f0c88d409594...,41.0
3,8147581512265419576,0,1078.65,4,526,14575487352341610617,14942,NaN,AUS,NaN,NaN,NaN,Mozilla/5.0 (iPhone; CPU iPhone OS 17_2_1 like...,NaN,6e23dbda72e8f0b5236187e648159b5cca78fb829b56f7...,43.0
4,4959538505037167822,0,201.15,4,98,14575487352341610617,14942,NaN,AUS,NaN,NaN,NaN,Mozilla/5.0 (iPhone; CPU iPhone OS 17_2 like M...,NaN,63428e9b0a8e0a80b98f02428261e1e24048201f3a74bb...,67.0


In [7]:
train['is_fraud'].value_counts(normalize=True)

is_fraud
0    0.960873
1    0.039127
Name: proportion, dtype: float64

In [8]:
train.dropna(inplace=True)

# Feature engineering 

In [11]:
def create_amount_features(df):
    # Transaction amount percentiles/stats per merchant
    df['amount_to_merchant_mean'] = df.groupby('merchant_id')['currency_amount'].transform('mean')
    df['amount_to_merchant_std'] = df.groupby('merchant_id')['currency_amount'].transform('std')
    
    # Ratio features
    df['amount_to_merchant_ratio'] = df['currency_amount'] / (df['amount_to_merchant_mean'] + 1e-8)

In [12]:
def create_merchant_features(df):
    # Merchant transaction frequency
    df['merchant_tx_count'] = df.groupby('merchant_id')['transaction_id'].transform('count')
    
    # Merchant-currency patterns
    df['merchant_currency_ratio'] = df.groupby(['merchant_id', 'currency_id'])['transaction_id'].transform('count') / df['merchant_tx_count']
    
    # Merchant shop patterns
    df['merchant_shop_ratio'] = df.groupby(['merchant_id', 'merchant_shop_id'])['transaction_id'].transform('count') / df['merchant_tx_count']

In [13]:
def create_card_features(df):
    # Card usage patterns
    df['card_merchant_count'] = df.groupby('card_exp_relative')['merchant_id'].transform('nunique')
    df['card_currency_count'] = df.groupby('card_exp_relative')['currency_id'].transform('nunique')
    
    # Card amount patterns
    df['card_amount_mean'] = df.groupby('card_exp_relative')['currency_amount'].transform('mean')
    df['card_amount_std'] = df.groupby('card_exp_relative')['currency_amount'].transform('std')

In [14]:
def encode_categorical_features(df):
    from sklearn.preprocessing import LabelEncoder
    
    # Target encoding for high-cardinality features
    df['browser_fraud_rate'] = df.groupby('browser')['is_fraud'].transform('mean')
    df['os_fraud_rate'] = df.groupby('operating_system')['is_fraud'].transform('mean')
    
    # Frequency encoding
    df['browser_freq'] = df.groupby('browser')['transaction_id'].transform('count') / len(df)
    df['os_freq'] = df.groupby('operating_system')['transaction_id'].transform('count') / len(df)

In [15]:
def create_interaction_features(df):
    # Interaction between amount and merchant features
    df['amount_merchant_interaction'] = df['currency_amount'] * df['merchant_tx_count']
    
    # Currency-amount interactions
    df['currency_amount_interaction'] = df['currency_id'].astype(str) + '_' + df['currency_amount'].astype(str)
    
    # Time-based interactions (if you have timestamp)
    # df['hour_amount_interaction'] = df['hour'] * df['amount_scaled']

In [16]:
def create_anomaly_features(df):
    from scipy import stats
    
    # Z-score for amounts
    df['amount_zscore'] = stats.zscore(df['currency_amount'])
    
    # Isolation Forest anomaly score
    from sklearn.ensemble import IsolationForest
    df['isolation_score'] = IsolationForest().fit_predict(df[['currency_amount', 'merchant_tx_count']])

In [17]:
def engineer_features(df):
    df = df.copy()
    
    create_amount_features(df)
    create_merchant_features(df)
    create_card_features(df)
    # encode_categorical_features(df)
    create_interaction_features(df)
    create_anomaly_features(df)
    
    return df

# Apply feature engineering
df_engineered = engineer_features(train)

# Training xgboost model 

In [ ]:
# def train_memory_efficient_xgboost(train_path, test_path, chunk_size=10000):
#     print("Starting memory-efficient training...")
    
#     # Read first chunk and get numeric features only
#     first_chunk = next(pd.read_csv(train_path, chunksize=chunk_size, low_memory=False))
    
#     # Explicitly select numeric columns and remove categorical ones
#     numeric_features = first_chunk.select_dtypes(include=[np.number]).columns.tolist()
#     exclude_columns = ['payment_type', 'transaction_source', 'browser', 
#                       'browser_version', 'operating_system', 
#                       'operating_system_version', 'is_fraud']
#     feature_names = [col for col in numeric_features if col not in exclude_columns]
    
#     print(f"Using numeric features only: {feature_names}")
    
#     del first_chunk
#     gc.collect()
    
#     # Initialize model with better parameters
#     model = xgb.XGBClassifier(
#         tree_method='hist',
#         max_depth=8,
#         learning_rate=0.05,
#         n_estimators=200,
#         min_child_weight=1,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         scale_pos_weight=10,  # Adjust for class imbalance
#         use_label_encoder=False,
#         n_jobs=-1
#     )
    
#     print("Training and evaluating on validation set...")
#     validation_predictions = []
#     validation_true = []
    
#     # Use last 5 chunks for validation
#     for i, chunk in enumerate(pd.read_csv(train_path, chunksize=chunk_size, low_memory=False)):
#         # Feature engineering
#         chunk['amount_to_order'] = chunk['amount_scaled'] / (chunk['order_number'] + 1)
#         chunk['currency_amount_scaled'] = chunk['currency_amount'] / chunk['amount_scaled']
        
#         if i < 45:  # First 45 chunks for training
#             X_chunk = chunk[feature_names]
#             y_chunk = chunk['is_fraud']
#             model.fit(X_chunk, y_chunk, xgb_model=model.get_booster() if i > 0 else None)
            
#             if i % 10 == 0:
#                 print(f"Processed {i} training chunks...")
                
#         elif i < 50:  # Last 5 chunks for validation
#             X_val = chunk[feature_names]
#             y_val = chunk['is_fraud']
#             y_pred = model.predict(X_val)
            
#             validation_predictions.extend(y_pred)
#             validation_true.extend(y_val)
            
#             print(f"Processed validation chunk {i}")
            
#         else:
#             break
            
#         del chunk
#         gc.collect()
    
#     # Calculate and print validation metrics
#     print("\nValidation Metrics:")
#     print(classification_report(validation_true, validation_predictions))
#     macro_f1 = f1_score(validation_true, validation_predictions, average='macro')
#     print(f"Validation Macro F1 Score: {macro_f1:.4f}")
    
#     print("\nMaking predictions on test data...")
#     predictions = []
#     transaction_ids = []
    
#     for i, test_chunk in enumerate(pd.read_csv(test_path, chunksize=chunk_size, low_memory=False)):
#         # Apply same feature engineering to test data
#         test_chunk['amount_to_order'] = test_chunk['amount_scaled'] / (test_chunk['order_number'] + 1)
#         test_chunk['currency_amount_scaled'] = test_chunk['currency_amount'] / test_chunk['amount_scaled']
        
#         X_test = test_chunk[feature_names]
#         y_pred = model.predict(X_test)
        
#         # Store transaction IDs and predictions
#         transaction_ids.extend(test_chunk['transaction_id'].tolist())
#         predictions.extend(y_pred.tolist())
        
#         del X_test, test_chunk
#         gc.collect()
        
#         if i % 10 == 0:
#             print(f"Processed {i} test chunks...")
    
#     # Create submission DataFrame
#     final_predictions = pd.DataFrame({
#         'transaction_id': transaction_ids,
#         'is_fraud': predictions
#     })
    
#     print(f"\nTotal predictions made: {len(final_predictions)}")
    
#     # Print feature importance
#     importance_df = pd.DataFrame({
#         'feature': feature_names,
#         'importance': model.feature_importances_
#     })
#     print("\nFeature Importance:")
#     print(importance_df.sort_values('importance', ascending=False))
    
#     return model, final_predictions, macro_f1

In [ ]:
# # Run training
# try:
#     print("Starting training process...")
#     model, predictions, macro_f1 = train_memory_efficient_xgboost(train_path, test_path)
    
#     # Save model
#     model.save_model('fraud_detection_model.json')
#     print("\nModel saved to fraud_detection_model.json")
    
#     # Save predictions in required format
#     predictions.to_csv('submission.csv', index=False)
#     print("Predictions saved to submission.csv")
    
#     # Print prediction statistics
#     print("\nPrediction Statistics:")
#     print(f"Number of fraudulent transactions predicted: {sum(predictions['is_fraud'] == 1)}")
#     print(f"Number of normal transactions predicted: {sum(predictions['is_fraud'] == 0)}")
#     print(f"Final Validation Macro F1 Score: {macro_f1:.4f}")
    
#     # Verify submission format
#     sample_predictions = predictions.head()
#     print("\nFirst few rows of submission file:")
#     print(sample_predictions)
    
# except Exception as e:
#     print(f"An error occurred: {str(e)}")
#     import traceback
#     print(traceback.format_exc())

rbgkjfnkjn

In [ ]:
def train_memory_efficient_xgboost(train_path, test_path, chunk_size=10000):
   print("Starting memory-efficient training...")
   
   # Read first chunk and get numeric features only
   first_chunk = next(pd.read_csv(train_path, chunksize=chunk_size, low_memory=False))
   
   # Explicitly select numeric columns and remove categorical ones
   numeric_features = first_chunk.select_dtypes(include=[np.number]).columns.tolist()
   exclude_columns = ['payment_type', 'transaction_source', 'browser', 
                     'browser_version', 'operating_system', 
                     'operating_system_version', 'is_fraud']
   feature_names = [col for col in numeric_features if col not in exclude_columns]
   
   print(f"Using numeric features only: {feature_names}")
   
   del first_chunk
   gc.collect()
   
   # Initialize model
   model = xgb.XGBClassifier(
       tree_method='hist',
       max_depth=6,
       learning_rate=0.1,
       n_estimators=100,
       use_label_encoder=False,
       n_jobs=-1
   )
   
   print("Training and evaluating on validation set...")
   validation_predictions = []
   validation_true = []
   
   # Use last 5 chunks for validation
   for i, chunk in enumerate(pd.read_csv(train_path, chunksize=chunk_size, low_memory=False)):
       if i < 45:  # First 45 chunks for training
           X_chunk = chunk[feature_names]
           y_chunk = chunk['is_fraud']
           model.fit(X_chunk, y_chunk, xgb_model=model.get_booster() if i > 0 else None)
           
           if i % 10 == 0:
               print(f"Processed {i} training chunks...")
               
       elif i < 50:  # Last 5 chunks for validation
           X_val = chunk[feature_names]
           y_val = chunk['is_fraud']
           y_pred = model.predict(X_val)
           
           validation_predictions.extend(y_pred)
           validation_true.extend(y_val)
           
           print(f"Processed validation chunk {i}")
           
       else:
           break
           
       # Free memory
       del chunk
       gc.collect()
   
   # Calculate and print validation metrics
   print("\nValidation Metrics:")
   print(classification_report(validation_true, validation_predictions))
   macro_f1 = f1_score(validation_true, validation_predictions, average='macro')
   print(f"Validation Macro F1 Score: {macro_f1:.4f}")
   
   print("\nMaking predictions on test data...")
   predictions = []
   
   for i, test_chunk in enumerate(pd.read_csv(test_path, chunksize=chunk_size, low_memory=False)):
       if i >= 10:
           break
           
       X_test = test_chunk[feature_names]
       y_pred = model.predict(X_test)
       
       # Create predictions DataFrame
       pred_df = pd.DataFrame({
           'prediction': y_pred
       })
       predictions.append(pred_df)
       
       del X_test, test_chunk
       gc.collect()
       
       if i % 10 == 0:
           print(f"Processed {i} test chunks...")
   
   # Combine all predictions
   final_predictions = pd.concat(predictions, axis=0, ignore_index=True)
   print(f"\nTotal predictions made: {len(final_predictions)}")
   
   # Print feature importance
   importance_df = pd.DataFrame({
       'feature': feature_names,
       'importance': model.feature_importances_
   })
   print("\nFeature Importance:")
   print(importance_df.sort_values('importance', ascending=False))
   
   return model, final_predictions, macro_f1

In [ ]:
try:
   print("Starting training process...")
   model, predictions, macro_f1 = train_memory_efficient_xgboost(train_path, test_path)
   
   # Save model
   model.save_model('fraud_detection_model.json')
   print("\nModel saved to fraud_detection_model.json")
   
   # Save predictions
   predictions.to_csv('test_predictions.csv', index=False)
   print("Predictions saved to test_predictions.csv")
   
   # Print prediction statistics
   print("\nPrediction Statistics:")
   print(f"Number of fraudulent transactions predicted: {sum(predictions['prediction'] == 1)}")
   print(f"Number of normal transactions predicted: {sum(predictions['prediction'] == 0)}")
   print(f"Final Validation Macro F1 Score: {macro_f1:.4f}")
   
except Exception as e:
   print(f"An error occurred: {str(e)}")
   import traceback
   print(traceback.format_exc())

gm,kfgmfkmg